# Model Evaluation
**ERROR ANALYSIS · MBE · LIVE SCENARIO**

---


## Inhalt

- [Setup](#setup)
- [Load Model & Predictions](#load-model-predictions)
- [Metrics — Model vs. Baseline](#metrics-model-vs-baseline)
- [Error Analysis](#error-analysis)
- [Key Findings — Fehleranalyse](#key-findings-fehleranalyse)
- [Improvement Opportunities](#improvement-opportunities)
- [Feature Importance — Was hat das Modell gelernt?](#feature-importance-was-hat-das-modell-gelernt)
- [Residuals — Systematischer Bias?](#residuals-systematischer-bias)
  - [Predicted vs. Actual — Bias sichtbar gemacht](#predicted-vs-actual-bias-sichtbar-gemacht)
- [Concrete Prediction — The Scenario from Overview](#concrete-prediction-the-scenario-from-overview)
- [Conclusion](#conclusion)
- [Summary — What the Model Shows](#summary-what-the-model-shows)


Wir laden die fertig berechneten Test-Predictions aus `06_prediction_2-model.ipynb` und analysieren wo das Modell gut und wo es schwächer ist.

**Benchmark:** Stop Mean Baseline MAE = 50.0s &nbsp;|&nbsp; **LightGBM v1 Test MAE = 45.7s** (−4.3s gegenüber Baseline ✅)

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import polars as pl
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("06_prediction_3-evaluation")


## Load Model & Predictions

In [ ]:
pred_path = Path(str(TEST)).parent / "test_predictions.parquet"
pred = pl.read_parquet(pred_path)

print(f"Predictions geladen: {len(pred):,} Zeilen")
print(f"Spalten: {pred.columns}")
pred.head(5)

## Metrics — Model vs. Baseline


In [ ]:
def mae(df, actual='actual', predicted='predicted'):
    return (df[actual] - df[predicted]).abs().mean()

def rmse(df, actual='actual', predicted='predicted'):
    return ((df[actual] - df[predicted]) ** 2).mean() ** 0.5

def otp(df, actual='actual', predicted='predicted', threshold=60):
    return ((df[actual] - df[predicted]).abs() <= threshold).mean()

# Modell-Metriken auf Test-Set
model_mae  = mae(pred)
model_rmse = rmse(pred)
model_otp  = otp(pred)

# Baseline-Werte aus 06_prediction_1-baseline.ipynb (Stop Mean)
BASELINE_MAE  = 50.0   # Stop Mean MAE
BASELINE_RMSE = 77.4   # Stop Mean RMSE
BASELINE_OTP  = 0.719  # Stop Mean OTP ±60s

results = pl.DataFrame({
    "":          ["Stop Mean Baseline", "LightGBM v1", "Gewinn"],
    "MAE (s)":   [BASELINE_MAE, round(model_mae, 1), round(BASELINE_MAE - model_mae, 1)],
    "RMSE (s)":  [BASELINE_RMSE, round(model_rmse, 1), round(BASELINE_RMSE - model_rmse, 1)],
    "OTP ±60s":  [f"{BASELINE_OTP:.1%}", f"{model_otp:.1%}", f"+{(model_otp - BASELINE_OTP):.1%}"],
})

show_df(results.to_pandas())

## Error Analysis

Wo liegt das Modell daneben? Wir schlüsseln den MAE auf nach:
- **Tageszeit** — Rush-Hour vs. Nacht
- **Linie** — welche Linien sind schwer vorherzusagen?
- **Wetter** — Schnee / Regen / normal
- **Monat** — saisonale Schwächen

In [ ]:
# --- MAE nach Stunde ---
mae_hour = (
    pred
    .with_columns((pl.col('actual') - pl.col('predicted')).abs().alias('abs_err'))
    .group_by('hour')
    .agg(pl.col('abs_err').mean().alias('MAE'), pl.len().alias('n'))
    .sort('hour')
)

fig = px.bar(mae_hour.to_pandas(), x='hour', y='MAE',
             title='MAE nach Tageszeit',
             labels={'hour': 'Stunde', 'MAE': 'MAE (s)'})
fig.add_hline(y=model_mae, line_dash='dash', line_color='gray',
              annotation_text=f'Gesamt MAE {model_mae:.1f}s')
fig.show()
show_df(mae_hour.sort('MAE', descending=True).to_pandas())

In [ ]:
# --- MAE nach Linie ---
mae_line = (
    pred
    .with_columns((pl.col('actual') - pl.col('predicted')).abs().alias('abs_err'))
    .group_by('line_name')
    .agg(pl.col('abs_err').mean().alias('MAE'), pl.len().alias('n'))
    .sort('MAE', descending=True)
)

fig = px.bar(mae_line.to_pandas(), x='line_name', y='MAE',
             title='MAE nach Linie',
             labels={'line_name': 'Linie', 'MAE': 'MAE (s)'})
fig.add_hline(y=model_mae, line_dash='dash', line_color='gray',
              annotation_text=f'Gesamt MAE {model_mae:.1f}s')
fig.show()
show_df(mae_line.to_pandas())

In [ ]:
# --- MAE nach Wetter ---
mae_weather = (
    pred
    .with_columns([
        (pl.col('actual') - pl.col('predicted')).abs().alias('abs_err'),
        pl.when(pl.col('has_snow')).then(pl.lit('Schnee'))
          .when(pl.col('has_rain')).then(pl.lit('Regen'))
          .otherwise(pl.lit('Normal')).alias('weather'),
    ])
    .group_by('weather')
    .agg(pl.col('abs_err').mean().alias('MAE'), pl.len().alias('n'))
    .sort('MAE', descending=True)
)

fig = px.bar(mae_weather.to_pandas(), x='weather', y='MAE',
             title='MAE nach Wetterbedingung',
             labels={'weather': 'Wetter', 'MAE': 'MAE (s)'})
fig.add_hline(y=model_mae, line_dash='dash', line_color='gray',
              annotation_text=f'Gesamt MAE {model_mae:.1f}s')
fig.show()
show_df(mae_weather.to_pandas())

In [ ]:
# --- MAE nach Monat ---
mae_month = (
    pred
    .with_columns((pl.col('actual') - pl.col('predicted')).abs().alias('abs_err'))
    .group_by('month')
    .agg(pl.col('abs_err').mean().alias('MAE'), pl.len().alias('n'))
    .sort('month')
)

fig = px.bar(mae_month.to_pandas(), x='month', y='MAE',
             title='MAE nach Monat (Test-Jahr 2025)',
             labels={'month': 'Monat', 'MAE': 'MAE (s)'})
fig.add_hline(y=model_mae, line_dash='dash', line_color='gray',
              annotation_text=f'Gesamt MAE {model_mae:.1f}s')
fig.show()
show_df(mae_month.to_pandas())

## Key Findings — Fehleranalyse

**Rush-Hour ist die stärkste Schwachstelle**
* 16h: 53.3s · 17h: 54.5s · 18h: 52.8s — alle deutlich über Gesamtschnitt (45.7s)
* Beste Stunden: 4–6h mit 33–40s — geringer Verkehr = geringere Varianz

**L11 und L8 schlagen die Baseline nicht**
* L11: 52.3s · L8: 51.1s · L15: 50.5s — alle über Baseline (50.0s)
* Genau die Linien durch K11/K12 — strukturell schwierigste Zonen des Netzes
* Gegenstück: L12 34.5s · L6 36.6s · L17 39.3s — größte Modell-Gewinne

**Schnee überfordert das Modell**
* MAE Schnee 58.9s vs. Normal 45.4s — Differenz +13.5s
* Nur 40k Schnee-Halte im Testjahr 2025 — zu wenig Trainings- und Testdaten für seltene Ereignisse

**Systematischer Optimismus-Bias**
* MBE +8.3s — Modell unterschätzt realen Delay durchgehend
* Besonders ausgeprägt bei Extremlagen: Rush-Hour, Schnee, Linie 11

---

## Improvement Opportunities

**Kurzfristig — gleicher Stack, mehr Signal**
* `prev_stop_delay` als Feature — Delay des Vorgänger-Halts in derselben Fahrt (Kaskadenindikator)
* `stop_sequence` explizit als numerisches Feature — Delay akkumuliert mit Strecke
* Baustellenkalender als binäres Feature — temporäre Streckenstörungen

**Mittelfristig — Modell-Tuning**
* Quantile Regression (`objective="quantile"`) statt MAE — reduziert Optimismus-Bias direkt
* Separate Modelle für L11 / L8 — linienspezifische Muster lernen
* Hyperparameter-Tuning mit Optuna (num_leaves, min_child_samples, learning_rate)

**Längerfristig — alternative Ansätze**
* XGBoost — robuster bei Extremwerten durch andere Baum-Regularisierung
* CatBoost — stärker bei hochkardinalen Kategorien wie `stop_name`
* Sequenzmodell (z.B. LightGBM mit Rolling Features) — explizite Zeitabhängigkeit modellieren

## Feature Importance — Was hat das Modell gelernt?

Hat das Modell dieselben Muster gelernt, die die 66 Findings der Analyse-Phase beschreiben?

Feature Importance (Gain) zeigt, wie viel jedes Feature zur Reduktion des Vorhersagefehlers beiträgt — normalisiert auf 100 %.

`stop_name` und `hour` sollten dominieren (räumlich + temporal stärkste Signale aus Analyse). `has_snow` und `is_holiday` zeigen ob der Wetter- und Event-Effekt gelernt wurde.

In [ ]:
import lightgbm as lgb
from wgnd.core.theme import mpl_style
from wgnd.core.config import cfg
import matplotlib.pyplot as plt

model_path = Path(str(TEST)).parent.parent / "models" / "lgbm_v1.txt"
model = lgb.Booster(model_file=str(model_path))

# Feature Importance — Gain (wie viel erklärt jedes Feature?)
feat_imp = (
    pd.DataFrame({
        "feature":    model.feature_name(),
        "gain":       model.feature_importance(importance_type="gain"),
        "split":      model.feature_importance(importance_type="split"),
    })
    .assign(pct=lambda df: df["gain"] / df["gain"].sum() * 100)
    .sort_values("pct", ascending=False)
    .reset_index(drop=True)
)

top = feat_imp.head(20).copy()

# Farb-Kodierung: Amber = dominierend (>10%), Teal = relevant (>2%), Grau = Rest
def _color(pct):
    if pct > 10:  return cfg.COLOR_SIGNAL
    if pct > 2:   return cfg.COLOR_POSITIVE
    return cfg.ANNO_REF

colors = [_color(p) for p in top["pct"]]

style = mpl_style()
fig, ax = plt.subplots(figsize=(10, 7))

bars = ax.barh(
    top["feature"][::-1], top["pct"][::-1],
    color=colors[::-1], edgecolor="white", linewidth=0.4,
)
for bar, pct in zip(bars, top["pct"][::-1]):
    ax.text(
        bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
        f"{pct:.1f}%", va="center", fontsize=9,
        color=cfg.CHART_AXIS_TEXT,
    )

ax.set_xlabel("Importance (% Gain)", **style["label"])
ax.set_title(
    "Feature Importance — LightGBM v1 (Gain)\n"
    "Amber > 10% · Teal > 2% · Grau < 2%",
    **style["title"],
)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
ax.tick_params(colors=cfg.CHART_AXIS_TEXT, labelsize=9)
ax.set_xlim(0, top["pct"].max() * 1.15)
plt.tight_layout()
plt.show()

show_df(
    feat_imp.head(33)
    [["feature", "pct", "split"]]
    .rename(columns={"feature": "Feature", "pct": "Gain (%)", "split": "Splits"})
    .round({"Gain (%)": 2})
    .reset_index(drop=True)
)

## Residuals — Systematischer Bias?

Schaut das Modell systematisch zu optimistisch (Vorhersage < Ist) oder zu pessimistisch (Vorhersage > Ist)?

**Mean Bias Error (MBE):** positiv = Modell überschätzt Delay · negativ = unterschätzt

In [ ]:
residuals = pred.with_columns(
    (pl.col('actual') - pl.col('predicted')).alias('residual')
)

mbe = residuals['residual'].mean()
print(f"Mean Bias Error (MBE): {mbe:.2f}s")
print(f"  > 0 = Modell unterschätzt Delay (zu optimistisch)")
print(f"  < 0 = Modell überschätzt Delay (zu pessimistisch)")

# Residual-Verteilung
sample = residuals.sample(n=min(50_000, len(residuals)), seed=42)
fig = px.histogram(sample.to_pandas(), x='residual', nbins=100,
                   title='Residual-Verteilung (actual − predicted)',
                   labels={'residual': 'Residual (s)', 'count': 'Anzahl'},
                   range_x=[-300, 300])
fig.add_vline(x=0, line_color='red', line_dash='dash')
fig.add_vline(x=mbe, line_color='orange',
              annotation_text=f'MBE {mbe:.1f}s')
fig.show()

### Predicted vs. Actual — Bias sichtbar gemacht

Hexbin-Dichte (100k Stichprobe): zeigt wo Vorhersagen konzentriert sind.
Ideal: alle Punkte auf der gestrichelten **y = x** Linie.
Der Bias (+8.3s) ist sichtbar als **Verschiebung nach unten** — Modell sagt systematisch weniger Delay voraus als tatsächlich eintritt.

In [ ]:
sample = pred.sample(n=100_000, seed=42).to_pandas()

CLIP_LO, CLIP_HI = -120, 500   # sinnvoller Darstellungsbereich (s)

style = mpl_style()
fig, ax = plt.subplots(figsize=(7, 7))

hb = ax.hexbin(
    sample["actual"].clip(CLIP_LO, CLIP_HI),
    sample["predicted"].clip(CLIP_LO, CLIP_HI),
    gridsize=60, cmap="YlOrRd", mincnt=1,
    extent=[CLIP_LO, CLIP_HI, CLIP_LO, CLIP_HI],
)

# Perfekte Vorhersage-Linie (y = x)
ax.plot([CLIP_LO, CLIP_HI], [CLIP_LO, CLIP_HI],
        color="#333333", lw=1.5, ls="--", label="Perfekte Vorhersage (y = x)", zorder=5)

# Bias-Linie (verschoben um MBE)
mbe_val = float(residuals["residual"].mean())
ax.plot([CLIP_LO, CLIP_HI], [CLIP_LO - mbe_val, CLIP_HI - mbe_val],
        color=cfg.COLOR_NEGATIVE, lw=1.5, ls="--",
        label=f"Modell-Bias (MBE +{mbe_val:.1f}s)", zorder=4)

ax.set_xlabel("Tatsächlicher Delay (s)", **style["label"])
ax.set_ylabel("Vorhergesagter Delay (s)", **style["label"])
ax.set_title("Predicted vs. Actual — LightGBM v1\n(Hexbin-Dichte · 100k Stichprobe)", **style["title"])
ax.legend(fontsize=9, frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
ax.tick_params(colors=cfg.CHART_AXIS_TEXT, labelsize=9)

cb = fig.colorbar(hb, ax=ax, fraction=0.03, pad=0.02)
cb.set_label("Anzahl Halte", fontsize=9)
cb.ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

## Concrete Prediction — The Scenario from Overview


Live-Demonstration: Eine einzelne Eingabe → eine Vorhersage. Input exakt wie im Szenario aus `06_prediction_0-overview`.

In [ ]:
import lightgbm as lgb

model_path = Path(str(TEST)).parent.parent / "models" / "lgbm_v1.txt"
model = lgb.Booster(model_file=str(model_path))

# Szenario aus 06_prediction_0-overview:
# Dienstag 17:00 · Haltestelle Paradeplatz · Linie 11 · leichter Regen · kein Event
scenario = pd.DataFrame([{
    "line_name":          "11",
    "stop_name":          "Paradeplatz",
    "district_nr":        1,
    "temperature":        14.0,
    "precipitation":      1.5,
    "wind_speed":         12.0,
    "flood_intensity":    0,
    "event_type":         "none",
    "event_size":         0,
    "hour":               17,
    "weekday":            1,
    "month":              6,
    "year":               2025,
    "season":             "summer",
    "is_weekend":         False,
    "is_november":        False,
    "gtfs_year":          "j25",
    "has_rain":           True,
    "has_heavy_rain":     False,
    "has_snow":           False,
    "has_flood":          False,
    "is_hot":             False,
    "is_holiday":         False,
    "has_event":          False,
    "event_weight":       0,
    "dwell_time":         0,
    "n_lines_at_stop":    14,
    "n_stops_line":       30,
    "is_start_stop":      False,
    "is_end_stop":        False,
    "event_weight_x_hour": 0,
    "is_late_night_weekend": False,
}])

# Kategoriale Spalten setzen
for col in ['line_name', 'stop_name', 'event_type', 'season', 'gtfs_year']:
    scenario[col] = scenario[col].astype('category')

pred_val = model.predict(scenario)[0]
print(f"Szenario: Dienstag 17:00 · Paradeplatz · Linie 11 · leichter Regen")
print(f"Vorhergesagter Delay: {pred_val:.0f}s ({pred_val/60:.1f} min)")

## Conclusion


In [ ]:
fazit = pl.DataFrame({
    "Modell":       ["Grand Mean", "Hour Mean", "Line Mean", "Stop Mean", "LightGBM v1"],
    "MAE (s)":      [50.6, 50.5, 50.4, 50.0, round(model_mae, 1)],
    "vs. Baseline": ["—", "—", "—", "Benchmark", f"-{BASELINE_MAE - model_mae:.1f}s ✅"],
})

show_df(fazit.to_pandas())

print()
print("Fazit:")
print(f"  LightGBM v1 erreicht MAE {model_mae:.1f}s auf dem Test-Set (2025).")
print(f"  Das Modell schlägt die Stop-Mean-Baseline ({BASELINE_MAE}s) um {BASELINE_MAE - model_mae:.1f}s.")
print(f"  Stärkste Schwäche: Rush-Hour und Schneetage (siehe Error Analysis).")

## Summary — What the Model Shows

**Das Modell funktioniert — und es ist ehrlich über seine Grenzen.**

**Was funktioniert**
* Baseline geschlagen: −4.3s MAE auf einem kompletten Testjahr (2025) mit 30 Mio. Halts
* Gute Generalisierung: Test MAE (45.7s) besser als Val MAE (49.05s) — kein Overfitting
* Linien ohne strukturelle Anomalie (L12, L6, L17): Gewinn bis zu −15s gegenüber Baseline
* Live-Vorhersage: Di 17h · Paradeplatz · L11 · Regen → **52s** — plausibel, direkt nutzbar

**Was schwierig bleibt**
* Rush-Hour (16–18h): Delay-Spitzen zu variabel für verlässliche Vorhersage
* L11 / L8: Zu hohe Varianz in den Problemzonen — Baseline nicht geschlagen
* Seltene Ereignisse (Schnee, Extremnacht): zu wenig Daten für robuste Schätzungen
* Systematischer Bias: +8.3s Unterschätzung — Modell ist zu optimistisch

**Was das für die Praxis bedeutet**
* Das Modell eignet sich gut für **typische Situationen** (80% der Halte): normale Tage, bekannte Linien
* Für **Extremlagen** (Rush-Hour, Schnee, L11) braucht es mehr Signal: Cascade-Features, separate Modelle
* Die Fehleranalyse zeigt klar **wo als nächstes** anzusetzen ist — das ist kein Zufallsfund, sondern direkte Ableitung aus den 66 Findings der Analyse-Phase

**Nächste Schritte**
* `prev_stop_delay` Feature hinzufügen → LightGBM v2
* Interaktives Vorhersage-Tool: Haltestelle + Linie + Uhrzeit + Wetter → Delay in Sekunden